# Median Runner

This notebook demonstrates how to create a grading workflow using PyBryt.

In [1]:
import pybryt

In [2]:
# ── PyBryt 0.7.0 Windows compatibility patch ──────────────────────────────────
#
# THREE BUGS FIXED
# ─────────────────
# Bug 1 — EOFError: mkstemp() on Windows returns a path with backslashes
#   (e.g. C:\Users\Temp\tmpXXXXXX). When embedded raw into a generated code
#   cell string, backslashes become Python escape sequences, corrupting the path.
#   The subprocess cell crashes silently, the footprint file is never written,
#   and dill.load() reads an empty file → EOFError.
#   Fix: replace backslashes with forward slashes before embedding.
#
# Bug 2 — PermissionError on os.remove(): mkstemp() returns (fd, path) where
#   fd is an open OS-level file descriptor. Windows locks files with open
#   handles, so the subprocess cannot write to it AND os.remove() fails after
#   reading. Fix: call os.close(fd) immediately after mkstemp().
#
# Bug 3 — Wrong patch target: pybryt.student._execute() calls execute_notebook
#   via a locally-bound name created by `from .execution import execute_notebook`.
#   Patching pybryt.execution.execute_notebook has no effect. Must patch
#   pybryt.student.execute_notebook instead.
#
# Bug 4 — si.steps does not exist in 0.7.0. Use si.footprint.num_steps.
# ──────────────────────────────────────────────────────────────────────────────

import pybryt.student as _student
from pybryt.preprocessors import NotebookPreprocessor
from pybryt.utils import make_secret
import nbformat as _nbformat
import os as _os, dill as _dill
from copy import deepcopy as _deepcopy
from tempfile import mkstemp as _mkstemp
from textwrap import dedent as _dedent
from nbconvert.preprocessors import ExecutePreprocessor as _EP

def _patched_execute_notebook(nb, nb_path, addl_filenames=[], timeout=1200):
    nb = _deepcopy(nb)
    preprocessor = NotebookPreprocessor()
    nb = preprocessor.preprocess(nb)

    # Bug 2 fix: close the fd immediately so Windows releases the file lock
    fd, footprint_fp = _mkstemp()
    _os.close(fd)

    # Bug 1 fix: forward slashes survive embedding into Python source strings
    safe_fp = footprint_fp.replace("\\", "/")

    nb_dir = _os.path.abspath(_os.path.split(nb_path)[0])
    secret = make_secret()
    ftv = f"frame_tracer_{secret}"

    first_cell = _nbformat.v4.new_code_cell(_dedent(f"""
        import inspect, sys
        from pybryt.execution import FrameTracer
        {ftv} = FrameTracer(inspect.currentframe())
        {ftv}.start_trace(addl_filenames={addl_filenames})
        %cd {nb_dir}
    """))

    last_cell = _nbformat.v4.new_code_cell(_dedent(f"""
        {ftv}.end_trace()
        footprint = {ftv}.get_footprint()
        footprint.filter_out_unpickleable_values()
        import dill
        with open("{safe_fp}", "wb+") as f:
            dill.dump(footprint, f)
    """))

    nb["cells"].insert(0, first_cell)
    nb["cells"].append(last_cell)
    _EP(timeout=timeout, allow_errors=True).preprocess(nb)

    with open(footprint_fp, "rb") as f:
        footprint = _dill.load(f)
    _os.remove(footprint_fp)
    footprint.add_imports(*preprocessor.get_imports())
    footprint.set_executed_notebook(nb)
    return footprint

# Bug 3 fix: patch the name in pybryt.student's namespace, not pybryt.execution
_student.execute_notebook = _patched_execute_notebook

This demo has the following directory structure. This notebook, `index.ipynb`, runs PyBryt, `median.ipynb` is the assignment reference implementation, and `submissions` contains notebooks with student code in them.

## Reference Implementations

If you have marked up a reference implementation, like the one in [`median.ipynb`](median.ipynb), you can load this reference using `pybryt.ReferenceImplementation.compile`. Because references are relatively static and can take some time to execute, you can pickle the reference implementations to a file with `pybryt.ReferenceImplementation.dump`.

In [3]:
ref = pybryt.ReferenceImplementation.compile("median.ipynb")
ref.dump()

To load a pickled reference implementation, use `pybryt.ReferenceImplementation.load`:

In [4]:
ref = pybryt.ReferenceImplementation.load("median.pkl")
ref

## Assessing Submissions

To use PyBryt for grading multiple submissions, you can build a reproducible grading pipeline for an arbitrary number of submissions. To grab the submission notebook paths, the cell below uses `glob.glob`.

In [5]:
from glob import glob
subms = sorted(glob("submissions/*.ipynb"))
subms

['submissions\\subm01.ipynb',
 'submissions\\subm02.ipynb',
 'submissions\\subm03.ipynb',
 'submissions\\subm04.ipynb',
 'submissions\\subm05.ipynb',
 'submissions\\subm06.ipynb']

To use PyBryt to grade a student's submission, a `pybryt.StudentImplementation` must be created from that submission. The constructor takes the path to the notebook as its only positional argument. It is in this step that the student's code is executed, so this cell will need to be rerun whenever changes are made to the submission notebooks.

In [6]:
student_impls = []
for subm in subms:
    student_impls.append(pybryt.StudentImplementation(subm))

student_impls

Once you have created the `pybryt.StudentImplementation` objects, use the `pybryt.StudentImplementation.check` method to run the check of a submission against a reference implementation. This method returns a single `pybryt.ReferenceResult` or a list of them, depending on the argument passed to `check`. In the cell below, the results are collected into a list.

In [7]:
results = []
for si in student_impls:
    results.append(si.check(ref))

results

[ReferenceResult([
   AnnotationResult(satisfied=True, annotation=pybryt.Value),
   AnnotationResult(satisfied=True, annotation=pybryt.Value),
   AnnotationResult(satisfied=True, annotation=pybryt.Value),
   AnnotationResult(satisfied=True, annotation=pybryt.Value),
   AnnotationResult(satisfied=True, annotation=pybryt.Value)
 ]),
 ReferenceResult([
   AnnotationResult(satisfied=False, annotation=pybryt.Value),
   AnnotationResult(satisfied=False, annotation=pybryt.Value),
   AnnotationResult(satisfied=False, annotation=pybryt.Value),
   AnnotationResult(satisfied=False, annotation=pybryt.Value),
   AnnotationResult(satisfied=True, annotation=pybryt.Value)
 ]),
 ReferenceResult([
   AnnotationResult(satisfied=True, annotation=pybryt.Value),
   AnnotationResult(satisfied=False, annotation=pybryt.Value),
   AnnotationResult(satisfied=True, annotation=pybryt.Value),
   AnnotationResult(satisfied=False, annotation=pybryt.Value),
   AnnotationResult(satisfied=True, annotation=pybryt.Value)


To view the results in a concise manner, the `pybryt.ReferenceResult` class has some helpful instance variables. You can also get information about the memory footprint, such as the number of steps, from the `pybryt.StudentImplementation` class.

In [8]:
from textwrap import indent
for sp, si, res in zip(subms, student_impls, results):
    print(f"SUBMISSION: {sp}")
    print(f"  EXECUTION STEPS: {si.footprint.num_steps}") # num_steps is on the footprint object in PyBryt 0.7.0

    # res.messages is a list of messages returned by the reference during grading
    messages = "\n".join(res.messages) 
    
    # res.correct is a boolean for whether the reference was satisfied
    message = f"SATISFIED: {res.correct}\nMESSAGES:\n{indent(messages, '  - ')}"
    
    # some pretty-printing
    print(indent(message, "  "))
    print("\n")

SUBMISSION: submissions\subm01.ipynb
  EXECUTION STEPS: 22
  SATISFIED: True
  MESSAGES:
    - SUCCESS: Sorted the sample correctly
    - SUCCESS: Computed the size of the sample
    - SUCCESS: Sorted the sample correctly
    - SUCCESS: Computed the size of the sample
    - computed the correct median


SUBMISSION: submissions\subm02.ipynb
  EXECUTION STEPS: 16
  SATISFIED: False
  MESSAGES:
    - ERROR: The sample was not sorted
    - ERROR: Did not capture the size of the set to determine if it is odd or even
    - ERROR: The sample was not sorted
    - ERROR: Did not capture the size of the set to determine if it is odd or even
    - computed the correct median


SUBMISSION: submissions\subm03.ipynb
  EXECUTION STEPS: 18
  SATISFIED: False
  MESSAGES:
    - SUCCESS: Sorted the sample correctly
    - ERROR: Did not capture the size of the set to determine if it is odd or even
    - SUCCESS: Sorted the sample correctly
    - ERROR: Did not capture the size of the set to determine if i

You can also turn the reference result objects into a JSON-friendly dictionary format for further processing:

In [9]:
res = results[0]
res.to_dict()

{'group': None,
 'results': [{'satisfied': True,
   'satisfied_at': 21,
   'annotation': {'name': 'Annotation 1',
    'group': None,
    'limit': None,
    'success_message': 'SUCCESS: Sorted the sample correctly',
    'failure_message': 'ERROR: The sample was not sorted',
    'children': [],
    'type': 'value',
    'invariants': [],
    'atol': None,
    'rtol': None},
   'children': []},
  {'satisfied': True,
   'satisfied_at': 16,
   'annotation': {'name': 'Annotation 2',
    'group': None,
    'limit': None,
    'success_message': 'SUCCESS: Computed the size of the sample',
    'failure_message': 'ERROR: Did not capture the size of the set to determine if it is odd or even',
    'children': [],
    'type': 'value',
    'invariants': [],
    'atol': None,
    'rtol': None},
   'children': []},
  {'satisfied': True,
   'satisfied_at': 21,
   'annotation': {'name': 'Annotation 3',
    'group': None,
    'limit': None,
    'success_message': 'SUCCESS: Sorted the sample correctly',
   